# Exhaustive Correlation Analysis: Logical Error Distribution vs Fixed-Class LLRs

This notebook performs **exhaustive correlation analysis** by exploring ALL logical classes (2^k)
for each shot. This provides complete data for analyzing whether the logical error distribution
predicts which error class will have the lowest fixed-class LLR.

**Key differences from sampled analysis:**
- **Complete coverage**: Explores all 2^k logical classes, not just sampled representatives
- **Parallelization**: Uses `num_procs_for_gap` for efficient parallel decoding
- **Fewer shots**: Due to computational cost, typically uses fewer shots than sampled mode

**Analyses in this notebook:**
1. **Data inspection**: Load and summarize the exhaustive exploration results
2. **Decoding success comparison**: Compare initial vs final success rates after gap proxy exploration
3. **Cluster statistics analysis**: Examine the relationship between error clustering and logical gap
4. **LLR visualization**: Visualize how fixed-class LLR varies with error rank across shots
5. **Second-best rank distribution**: Analyze which class ranks tend to be the runner-up
6. **Random gap proxy analysis**: Compare random sampling proxy to true gap as a function of N_classes and coverage_fraction

## 1. Data Collection Configuration

Configure the simulation parameters below, then run the cell to generate the script command.

In [23]:
# ==============================================================================
# CONFIGURATION - Set your simulation parameters here
# ==============================================================================

# Code parameters
N_QUBITS = 144          # BB code variant: 72, 90, 108, 144, 288, 360, 756
P = 0.003               # Physical error rate

# Simulation parameters (exhaustive mode)
N_SHOTS = 1000           # Number of shots to simulate (keep small, exhaustive is expensive)
NUM_PROCS_FOR_GAP = 126 # Parallel processes for gap computation
PARALLEL_VERBOSE = 3   # Joblib verbosity (0=silent, higher=more progress info)
SEED = 42               # Random seed

# Output path (auto-generated based on parameters)
OUTPUT_DIR = "simulations/data/correlation_analysis"

In [24]:
# ==============================================================================
# Generate script command based on configuration
# ==============================================================================

from pathlib import Path

# Generate output filename (parquet format, no shots in name for restart support)
OUTPUT_FILENAME = f"bb_n{N_QUBITS}_p{P}_exhaustive.parquet"
OUTPUT_PATH = Path(OUTPUT_DIR) / OUTPUT_FILENAME

# Build command
SCRIPT_PATH = "simulations/analysis/scripts/run_distribution_correlation_analysis.py"

cmd_parts = [
    f"python {SCRIPT_PATH}",
    f"    --n-qubits {N_QUBITS}",
    f"    --p {P}",
    f"    --n-shots {N_SHOTS}",
    f"    --exhaustive",
    f"    --num-procs-for-gap {NUM_PROCS_FOR_GAP}",
    f"    --parallel-verbose {PARALLEL_VERBOSE}",
    f"    --seed {SEED}",
    f"    --output {OUTPUT_PATH}",
]

command = " \\\n".join(cmd_parts)

# Display
print("=" * 70)
print("GENERATED COMMAND (Exhaustive Mode)")
print("=" * 70)
print()
print(command)
print()
print("=" * 70)
print(f"Output file: {OUTPUT_PATH}")
print("Note: If the output file exists, simulation will resume from last completed shot.")
print("=" * 70)

# Store for later use
RESULTS_PATH = Path("../../..") / OUTPUT_PATH

GENERATED COMMAND (Exhaustive Mode)

python simulations/analysis/scripts/run_distribution_correlation_analysis.py \
    --n-qubits 144 \
    --p 0.003 \
    --n-shots 1000 \
    --exhaustive \
    --num-procs-for-gap 126 \
    --parallel-verbose 3 \
    --seed 42 \
    --output simulations/data/correlation_analysis/bb_n144_p0.003_exhaustive.parquet

Output file: simulations/data/correlation_analysis/bb_n144_p0.003_exhaustive.parquet
Note: If the output file exists, simulation will resume from last completed shot.


## 2. Setup

Import required libraries and configure display settings.

In [25]:
# Autoreload extension for Jupyter notebooks
%load_ext autoreload
%autoreload 2

# Standard library imports
import json
from pathlib import Path

# Third-party library imports
import numpy as np
import pandas as pd
from scipy import stats

# Plotly for interactive visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pandas display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.6f}'.format)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 3. Load and Inspect Data

Load the simulation results from the Parquet file generated by the data collection script.

In [26]:
# ==============================================================================
# Load results (uses RESULTS_PATH from configuration above)
# ==============================================================================

if not RESULTS_PATH.exists():
    print(f"Results file not found: {RESULTS_PATH.resolve()}")
    print("\nPlease run the data collection script first using the command above.")
    raise FileNotFoundError(f"Results not found at {RESULTS_PATH}")

# Load DataFrame from Parquet
df_results = pd.read_parquet(RESULTS_PATH)

# Load metadata
metadata_path = RESULTS_PATH.with_suffix(".json")
if metadata_path.exists():
    with open(metadata_path) as f:
        metadata = json.load(f)
else:
    metadata = {}

# Display summary
print(f"Loaded {len(df_results):,} records from {RESULTS_PATH.name}")
print(f"\nMetadata:")
for key, value in metadata.items():
    if isinstance(value, dict):
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {value}")

print(f"\nDataFrame preview:")
df_results.head(10)

Loaded 2,306,048 records from bb_n144_p0.003_exhaustive.parquet

Metadata:

DataFrame preview:


,shot_id,error_rank,error_index,error_prob,fixed_llr,best_llr,llr_delta,initial_success,final_success,cluster_frac_llr_2norm
0,0,0,0,0.990456,217.045551,217.045551,0.000000,True,True,0.003410
1,0,302,3894,0.000006,411.639927,217.045551,194.594376,True,True,0.003410
2,0,83,1846,0.000022,421.962324,217.045551,204.916773,True,True,0.003410
3,0,3193,2870,0.000000,326.396211,217.045551,109.350660,True,True,0.003410
4,0,3605,822,0.000000,382.791794,217.045551,165.746243,True,True,0.003410
5,0,3600,3382,0.000000,408.243759,217.045551,191.198208,True,True,0.003410
6,0,2723,1334,0.000001,377.785936,217.045551,160.740385,True,True,0.003410
7,0,3892,2358,0.000000,344.302888,217.045551,127.257337,True,True,0.003410
8,0,733,310,0.000002,397.749723,217.045551,180.704172,True,True,0.003410
9,0,368,3638,0.000004,333.039252,217.045551,115.993701,True,True,0.003410


## 4. Data Summary

Overview of the loaded dataset including the number of shots, logical classes per shot,
and the structure of the exhaustive exploration data.

In [27]:
# ==============================================================================
# Data Summary
# ==============================================================================

n_shots = df_results['shot_id'].nunique()
n_classes_per_shot = len(df_results) // n_shots
total_logical_classes = metadata.get('total_logical_classes', n_classes_per_shot)

# Check for no-error case (error_index=0)
has_no_error = (df_results['error_index'] == 0).any()

# Find rank of no-error class
if has_no_error:
    no_error_rank = df_results[df_results['error_index'] == 0]['error_rank'].iloc[0]
    no_error_prob = df_results[df_results['error_index'] == 0]['error_prob'].iloc[0]
else:
    no_error_rank = None
    no_error_prob = None

print("=" * 60)
print("DATA SUMMARY")
print("=" * 60)
print(f"\nTotal shots: {n_shots}")
print(f"Classes per shot: {n_classes_per_shot}")
print(f"Total logical classes: {total_logical_classes}")
print(f"Total records: {len(df_results):,}")
print(f"\nNo-error class (index=0) included: {'Yes' if has_no_error else 'No'}")
if has_no_error:
    print(f"  - Rank: {no_error_rank} (0 = most common)")
    print(f"  - Probability: {no_error_prob:.4%}")
print(f"\nClass rank range: {df_results['error_rank'].min()} to {df_results['error_rank'].max()}")
print(f"LLR range: {df_results['fixed_llr'].min():.2f} to {df_results['fixed_llr'].max():.2f}")

DATA SUMMARY

Total shots: 563
Classes per shot: 4096
Total logical classes: 4096
Total records: 2,306,048

No-error class (index=0) included: Yes
  - Rank: 0 (0 = most common)
  - Probability: 99.0456%

Class rank range: 0 to 4095
LLR range: 94.71 to 1887.22


## 5. Decoding Success Comparison

Compare the initial BP+LSD decoding success rate with the final success rate after
exhaustive gap proxy exploration. This shows whether exploring all logical classes
improves decoding accuracy.

In [28]:
# ==============================================================================
# Decoding Success Statistics (if available)
# ==============================================================================

if "initial_success" in df_results.columns and "final_success" in df_results.columns:
    # Get per-shot success (same for all records in a shot)
    shot_success = df_results.groupby("shot_id").agg({
        "initial_success": "first",
        "final_success": "first",
    })
    
    initial_success_rate = shot_success["initial_success"].mean()
    final_success_rate = shot_success["final_success"].mean()
    
    # Count improvements and regressions
    improved = ((~shot_success["initial_success"]) & shot_success["final_success"]).sum()
    regressed = (shot_success["initial_success"] & (~shot_success["final_success"])).sum()
    unchanged = (shot_success["initial_success"] == shot_success["final_success"]).sum()
    
    print("=" * 60)
    print("DECODING SUCCESS COMPARISON")
    print("=" * 60)
    print(f"\nInitial BP+LSD success rate: {100*initial_success_rate:.2f}% ({int(shot_success['initial_success'].sum())}/{n_shots})")
    print(f"Final (comparative) success rate: {100*final_success_rate:.2f}% ({int(shot_success['final_success'].sum())}/{n_shots})")
    print(f"\nGap proxy exploration effect:")
    print(f"  Improved (wrong → correct): {improved} shots")
    print(f"  Regressed (correct → wrong): {regressed} shots")
    print(f"  Unchanged: {unchanged} shots")
    
    if initial_success_rate > 0:
        improvement_factor = final_success_rate / initial_success_rate
        print(f"\nImprovement factor: {improvement_factor:.2f}x")
else:
    print("Note: Decoding success columns not found in data.")

DECODING SUCCESS COMPARISON

Initial BP+LSD success rate: 98.58% (555/563)
Final (comparative) success rate: 99.82% (562/563)

Gap proxy exploration effect:
  Improved (wrong → correct): 8 shots
  Regressed (correct → wrong): 1 shots
  Unchanged: 554 shots

Improvement factor: 1.01x


## 6. Cluster Statistics vs Logical Gap

Analyze the relationship between cluster fraction LLR 2-norm (a measure of error clustering)
and the logical gap (difference between best and second-best LLR). Higher cluster fraction
indicates errors are more concentrated in clusters, which may correlate with decoder confidence.

In [29]:
# ==============================================================================
# Cluster Fraction LLR 2-Norm vs Logical Gap Analysis
# ==============================================================================

if "cluster_frac_llr_2norm" in df_results.columns:
    # Compute per-shot statistics including logical gap
    shot_stats_with_cluster = []
    
    for shot_id, group in df_results.groupby("shot_id"):
        # Get cluster fraction (same for all records in shot)
        cluster_frac = group["cluster_frac_llr_2norm"].iloc[0]
        best_llr = group["best_llr"].iloc[0]
        
        # Compute logical gap: difference between best and second-best LLR
        sorted_llrs = group["fixed_llr"].sort_values()
        min_llr = sorted_llrs.iloc[0]
        second_min_llr = sorted_llrs.iloc[1] if len(sorted_llrs) > 1 else min_llr
        logical_gap = second_min_llr - min_llr
        
        # Also get success info if available
        initial_success = group["initial_success"].iloc[0] if "initial_success" in group.columns else None
        
        shot_stats_with_cluster.append({
            "shot_id": shot_id,
            "cluster_frac_llr_2norm": cluster_frac,
            "best_llr": best_llr,
            "logical_gap": logical_gap,
            "min_llr": min_llr,
            "initial_success": initial_success,
        })
    
    df_cluster_gap = pd.DataFrame(shot_stats_with_cluster)
    
    print("=" * 60)
    print("CLUSTER FRACTION LLR 2-NORM vs LOGICAL GAP")
    print("=" * 60)
    
    print(f"\nCluster Fraction LLR 2-Norm:")
    print(f"  Mean:   {df_cluster_gap['cluster_frac_llr_2norm'].mean():.4f}")
    print(f"  Std:    {df_cluster_gap['cluster_frac_llr_2norm'].std():.4f}")
    print(f"  Range:  [{df_cluster_gap['cluster_frac_llr_2norm'].min():.4f}, {df_cluster_gap['cluster_frac_llr_2norm'].max():.4f}]")
    
    print(f"\nLogical Gap (2nd best LLR - best LLR):")
    print(f"  Mean:   {df_cluster_gap['logical_gap'].mean():.2f}")
    print(f"  Std:    {df_cluster_gap['logical_gap'].std():.2f}")
    print(f"  Range:  [{df_cluster_gap['logical_gap'].min():.2f}, {df_cluster_gap['logical_gap'].max():.2f}]")
    
    # Correlation analysis
    if len(df_cluster_gap) > 2:
        rho, p_val = stats.spearmanr(
            df_cluster_gap["cluster_frac_llr_2norm"], 
            df_cluster_gap["logical_gap"]
        )
        pearson_r, pearson_p = stats.pearsonr(
            df_cluster_gap["cluster_frac_llr_2norm"], 
            df_cluster_gap["logical_gap"]
        )
        
        print(f"\nCorrelation (Cluster Fraction vs Logical Gap):")
        print(f"  Spearman ρ: {rho:.4f} (p = {p_val:.2e})")
        print(f"  Pearson r:  {pearson_r:.4f} (p = {pearson_p:.2e})")
    
    # Scatter plot
    fig = go.Figure()
    
    # Color by initial_success if available
    if df_cluster_gap["initial_success"].notna().any():
        colors = df_cluster_gap["initial_success"].map({True: "green", False: "red"})
        color_labels = df_cluster_gap["initial_success"].map({True: "Success", False: "Fail"})
        
        for success_val, color, label in [(True, "green", "Initial Success"), (False, "red", "Initial Fail")]:
            mask = df_cluster_gap["initial_success"] == success_val
            if mask.any():
                fig.add_trace(go.Scatter(
                    x=df_cluster_gap.loc[mask, "cluster_frac_llr_2norm"],
                    y=df_cluster_gap.loc[mask, "logical_gap"],
                    mode="markers",
                    name=label,
                    marker=dict(color=color, size=12),
                    text=df_cluster_gap.loc[mask, "shot_id"],
                    hovertemplate="Shot %{text}<br>Cluster Frac: %{x:.4f}<br>Gap: %{y:.2f}<extra></extra>",
                ))
    else:
        fig.add_trace(go.Scatter(
            x=df_cluster_gap["cluster_frac_llr_2norm"],
            y=df_cluster_gap["logical_gap"],
            mode="markers",
            marker=dict(color="steelblue", size=12),
            text=df_cluster_gap["shot_id"],
            hovertemplate="Shot %{text}<br>Cluster Frac: %{x:.4f}<br>Gap: %{y:.2f}<extra></extra>",
        ))
    
    # Add correlation annotation
    if len(df_cluster_gap) > 2:
        fig.add_annotation(
            text=f"Spearman ρ = {rho:.3f}<br>p = {p_val:.2e}",
            xref="paper", yref="paper",
            x=0.02, y=0.98,
            showarrow=False,
            font=dict(size=12),
            bgcolor="white",
            bordercolor="gray",
            borderwidth=1,
            align="left",
        )
    
    fig.update_layout(
        title="Cluster Fraction LLR 2-Norm vs Logical Gap",
        xaxis_title="Cluster Fraction LLR 2-Norm (higher = more clustered)",
        yaxis_title="Logical Gap (2nd best - best LLR)",
        showlegend=True,
    )
    
    fig.show()
    
else:
    print("Note: cluster_frac_llr_2norm column not found in data.")

CLUSTER FRACTION LLR 2-NORM vs LOGICAL GAP

Cluster Fraction LLR 2-Norm:
  Mean:   0.0029
  Std:    0.0046
  Range:  [0.0004, 0.0397]

Logical Gap (2nd best LLR - best LLR):
  Mean:   60.65
  Std:    18.88
  Range:  [2.21, 106.78]

Correlation (Cluster Fraction vs Logical Gap):
  Spearman ρ: -0.4965 (p = 2.22e-36)
  Pearson r:  -0.4143 (p = 9.31e-25)


## 7. LLR vs Error Rank Visualization

Visualize how fixed-class LLR varies with error rank, aggregated across all shots.
The plot shows the median LLR at each rank with an interquartile range (IQR) band.

- **Rank 0**: Most probable error class (typically the no-error class at low error rates)
- **Higher ranks**: Less probable error classes
- **Star marker**: Highlights the no-error class (error_index=0)

In [30]:
# ==============================================================================
# LLR vs Logical Error Rank (aggregated across shots)
# ==============================================================================

# Compute Q1, Q2 (median), Q3 for each error rank
llr_by_rank = df_results.groupby("error_rank")["fixed_llr"].agg(
    Q1=lambda x: x.quantile(0.25),
    Q2=lambda x: x.quantile(0.50),
    Q3=lambda x: x.quantile(0.75),
).reset_index()

fig = go.Figure()

# Error band (Q1 to Q3)
fig.add_trace(go.Scatter(
    x=np.concatenate([llr_by_rank["error_rank"], llr_by_rank["error_rank"][::-1]]),
    y=np.concatenate([llr_by_rank["Q3"], llr_by_rank["Q1"][::-1]]),
    fill="toself",
    fillcolor="rgba(70, 130, 180, 0.3)",
    line=dict(width=0),
    name="IQR (Q1-Q3)",
    hoverinfo="skip",
))

# Median line (Q2)
fig.add_trace(go.Scatter(
    x=llr_by_rank["error_rank"],
    y=llr_by_rank["Q2"],
    mode="lines",
    name="Median (Q2)",
    line=dict(color="steelblue", width=2),
    hovertemplate="Rank: %{x}<br>Median LLR: %{y:.2f}<extra></extra>",
))

# Mark the no-error class (rank=0) with a star
no_error_stats = llr_by_rank[llr_by_rank["error_rank"] == 0]
if len(no_error_stats) > 0:
    fig.add_trace(go.Scatter(
        x=no_error_stats["error_rank"],
        y=no_error_stats["Q2"],
        mode="markers",
        marker=dict(color="gold", size=14, symbol="star", line=dict(width=1, color="black")),
        name="No-error class",
        hovertemplate="No error (rank=0)<br>Median LLR: %{y:.2f}<extra></extra>",
    ))

fig.update_layout(
    title="Fixed-Class LLR vs Logical Class Rank (Median ± IQR across Shots)",
    xaxis_title="Class Rank (0 = most common class from distribution)",
    yaxis_title="Fixed-Class LLR",
    hovermode="closest",
)

fig.show()

In [31]:
# ==============================================================================
# LLR vs Error Probability (aggregated across shots)
# ==============================================================================

# Compute Q1, Q2 (median), Q3 for each error probability
llr_by_prob = df_results.groupby("error_prob")["fixed_llr"].agg(
    Q1=lambda x: x.quantile(0.25),
    Q2=lambda x: x.quantile(0.50),
    Q3=lambda x: x.quantile(0.75),
).reset_index()

# Sort by probability for proper line plotting
llr_by_prob = llr_by_prob.sort_values("error_prob")

fig = go.Figure()

# Error band (Q1 to Q3)
fig.add_trace(go.Scatter(
    x=np.concatenate([llr_by_prob["error_prob"], llr_by_prob["error_prob"][::-1]]),
    y=np.concatenate([llr_by_prob["Q3"], llr_by_prob["Q1"][::-1]]),
    fill="toself",
    fillcolor="rgba(70, 130, 180, 0.3)",
    line=dict(width=0),
    name="IQR (Q1-Q3)",
    hoverinfo="skip",
))

# Median line (Q2)
fig.add_trace(go.Scatter(
    x=llr_by_prob["error_prob"],
    y=llr_by_prob["Q2"],
    mode="lines",
    name="Median (Q2)",
    line=dict(color="steelblue", width=2),
    hovertemplate="Prob: %{x:.2e}<br>Median LLR: %{y:.2f}<extra></extra>",
))

# Mark the no-error class (highest probability) with a star
no_error_prob = df_results[df_results["error_index"] == 0]["error_prob"].iloc[0]
no_error_stats = llr_by_prob[llr_by_prob["error_prob"] == no_error_prob]
if len(no_error_stats) > 0:
    fig.add_trace(go.Scatter(
        x=no_error_stats["error_prob"],
        y=no_error_stats["Q2"],
        mode="markers",
        marker=dict(color="gold", size=14, symbol="star", line=dict(width=1, color="black")),
        name="No-error class",
        hovertemplate="No error<br>Prob: %{x:.2e}<br>Median LLR: %{y:.2f}<extra></extra>",
    ))

fig.update_layout(
    title="Fixed-Class LLR vs Error Probability (Median ± IQR across Shots)",
    xaxis_title="Error Probability (from distribution)",
    xaxis_type="log",
    yaxis_title="Fixed-Class LLR",
    hovermode="closest",
)

fig.show()

## 8. Second-Best Class Rank Distribution

Visualize which logical class ranks tend to have the second-smallest LLR (the "runner-up").
This reveals whether the second-best decoding candidate follows a predictable pattern
based on the error distribution ranking.

Two complementary views are provided:
1. **By rank**: Distribution of the second-best class's rank in the error distribution
2. **By cumulative probability**: Cumulative probability mass up to the second-best rank

If the cumulative probability is close to 1, it means the runner-up is among the high-probability
classes that together account for most of the probability mass.

In [32]:
# ==============================================================================
# Second-Best Class Rank Distribution
# ==============================================================================

# Find the rank of the class with the second-smallest LLR for each shot
second_best_ranks = []

for shot_id, group in df_results.groupby("shot_id"):
    # Sort by LLR to find the second-smallest
    sorted_by_llr = group.sort_values("fixed_llr")
    if len(sorted_by_llr) >= 2:
        second_best_rank = sorted_by_llr.iloc[1]["error_rank"]
        second_best_ranks.append({
            "shot_id": shot_id,
            "second_best_rank": int(second_best_rank),
        })

df_second_best = pd.DataFrame(second_best_ranks)

# Print summary statistics
print("=" * 60)
print("SECOND-BEST CLASS RANK DISTRIBUTION")
print("=" * 60)
print(f"\nNumber of shots: {len(df_second_best)}")

print(f"\nSecond-best rank statistics:")
print(f"  Mean:   {df_second_best['second_best_rank'].mean():.1f}")
print(f"  Median: {df_second_best['second_best_rank'].median():.1f}")
print(f"  Std:    {df_second_best['second_best_rank'].std():.1f}")
print(f"  Min:    {df_second_best['second_best_rank'].min()}")
print(f"  Max:    {df_second_best['second_best_rank'].max()}")

# Create visualization: histogram
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df_second_best["second_best_rank"],
    nbinsx=500,
    marker=dict(color="steelblue", line=dict(width=1, color="black")),
    hovertemplate="Rank: %{x}<br>Count: %{y}<extra></extra>",
))

# Add a vertical line at the median
median_rank = df_second_best["second_best_rank"].median()
fig.add_vline(
    x=median_rank,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Median: {median_rank:.0f}",
    annotation_position="top",
)

fig.update_layout(
    title="Distribution of Second-Best Class Rank (Runner-Up)",
    xaxis_title="Class Rank (0 = most probable from distribution)",
    yaxis_title="Count",
    bargap=0.1,
)

fig.show()

SECOND-BEST CLASS RANK DISTRIBUTION

Number of shots: 563

Second-best rank statistics:
  Mean:   343.5
  Median: 112.0
  Std:    625.2
  Min:    0
  Max:    4040


In [33]:
# ==============================================================================
# Second-Best Class: Cumulative Probability (Calibrated, Excluding No-Error)
# ==============================================================================

# For each shot, compute the cumulative probability at the second-best class's rank
# using a calibrated distribution that excludes the no-error case (error_index=0)

second_best_cumprob = []

for shot_id, group in df_results.groupby("shot_id"):
    # Sort by LLR to find the second-smallest
    sorted_by_llr = group.sort_values("fixed_llr")
    if len(sorted_by_llr) >= 2:
        second_best_row = sorted_by_llr.iloc[1]
        second_best_rank = int(second_best_row["error_rank"])
        
        # Create calibrated distribution: exclude no-error case and renormalize
        error_classes = group[group["error_index"] != 0].copy()
        total_error_prob = error_classes["error_prob"].sum()
        error_classes["calibrated_prob"] = error_classes["error_prob"] / total_error_prob
        
        # Recompute ranks within the error-only distribution
        error_classes = error_classes.sort_values("calibrated_prob", ascending=False)
        error_classes["calibrated_rank"] = range(len(error_classes))
        
        # Find the second-best class in the calibrated distribution
        second_best_calibrated = error_classes[error_classes["error_index"] == second_best_row["error_index"]]
        
        if len(second_best_calibrated) > 0:
            calibrated_rank = int(second_best_calibrated["calibrated_rank"].iloc[0])
            # Cumulative probability: sum of calibrated probs for ranks 0 to calibrated_rank
            cumulative_prob = error_classes[error_classes["calibrated_rank"] <= calibrated_rank]["calibrated_prob"].sum()
            
            second_best_cumprob.append({
                "shot_id": shot_id,
                "original_rank": second_best_rank,
                "calibrated_rank": calibrated_rank,
                "cumulative_prob": cumulative_prob,
            })

df_second_best_cumprob = pd.DataFrame(second_best_cumprob)

# Print summary statistics
print("=" * 60)
print("SECOND-BEST CLASS: CALIBRATED CUMULATIVE PROBABILITY")
print("(Excluding no-error case, renormalized)")
print("=" * 60)
print(f"\nNumber of shots: {len(df_second_best_cumprob)}")

print(f"\nCumulative probability statistics:")
print(f"  Mean:   {df_second_best_cumprob['cumulative_prob'].mean():.4f}")
print(f"  Median: {df_second_best_cumprob['cumulative_prob'].median():.4f}")
print(f"  Std:    {df_second_best_cumprob['cumulative_prob'].std():.4f}")
print(f"  Min:    {df_second_best_cumprob['cumulative_prob'].min():.4f}")
print(f"  Max:    {df_second_best_cumprob['cumulative_prob'].max():.4f}")

# Create visualization: histogram
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df_second_best_cumprob["cumulative_prob"],
    nbinsx=100,
    marker=dict(color="darkorange", line=dict(width=1, color="black")),
    hovertemplate="Cumulative Prob: %{x:.3f}<br>Count: %{y}<extra></extra>",
))

# Add a vertical line at the median
median_cumprob = df_second_best_cumprob["cumulative_prob"].median()
fig.add_vline(
    x=median_cumprob,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Median: {median_cumprob:.3f}",
    annotation_position="top",
)

fig.update_layout(
    title="Cumulative Probability at Second-Best Class (Calibrated Distribution)",
    xaxis_title="Cumulative Probability (excluding no-error, renormalized)",
    xaxis=dict(range=[0, 1]),
    yaxis_title="Count",
    bargap=0.1,
)

fig.show()

SECOND-BEST CLASS: CALIBRATED CUMULATIVE PROBABILITY
(Excluding no-error case, renormalized)

Number of shots: 562

Cumulative probability statistics:
  Mean:   0.3775
  Median: 0.3341
  Std:    0.2319
  Min:    0.0050
  Max:    1.0000


## 9. Random Gap Proxy vs True Logical Gap Analysis

Analyze how well the "random" method approximates the true logical gap as a function of:
- **N_classes**: Number of classes explored (including initial best)
- **coverage_fraction**: Cumulative probability threshold for eligible error classes

The random method:
1. Starts with the **no-error class** (error_index=0, rank=0) as the initial best
2. Samples `N_classes - 1` additional classes from the **calibrated distribution** (error classes only, renormalized) within the coverage threshold
3. Computes the gap within the sampled subset (no-error + sampled error classes)

When `coverage_fraction=1.0` and `N_classes` equals all error classes + 1, the gap proxy should converge to the true gap.

In [34]:
# ==============================================================================
# Simulation Function: Random Gap Proxy
# ==============================================================================

def simulate_random_gap_proxy(
    shot_data: pd.DataFrame,
    n_classes: int,
    coverage_fraction: float,
    n_reps: int = 50,
    rng: np.random.Generator | None = None,
) -> np.ndarray:
    """
    Simulate the random gap proxy method for a single shot.
    
    The method:
    1. Starts with the no-error class (error_index=0) as initial best
    2. Samples N_classes-1 additional classes from the calibrated distribution
       (error classes only, excluding no-error) within coverage_fraction
    3. Computes gap within the sampled subset
    
    Special case: If N_classes >= num_eligible_error_classes + 1, all eligible
    classes are included and the gap is computed deterministically (no sampling).
    
    Parameters
    ----------
    shot_data : pd.DataFrame
        Data for a single shot with columns: error_index, error_prob, fixed_llr
    n_classes : int
        Total number of classes to explore (including initial best = no-error)
    coverage_fraction : float
        Cumulative probability threshold for eligible error classes (0, 1]
    n_reps : int
        Number of Monte Carlo repetitions (ignored if deterministic)
    rng : np.random.Generator or None
        Random number generator for reproducibility
    
    Returns
    -------
    gap_proxies : 1D numpy array of float
        Gap proxy values for each MC repetition (all identical if deterministic)
    """
    if rng is None:
        rng = np.random.default_rng()
    
    # Initial best class: no-error class (error_index=0)
    no_error_class = shot_data[shot_data["error_index"] == 0]
    if len(no_error_class) == 0:
        raise ValueError("No-error class (error_index=0) not found in shot_data")
    initial_best_llr = no_error_class["fixed_llr"].iloc[0]
    
    # Build calibrated distribution: exclude no-error case (error_index=0)
    error_classes = shot_data[shot_data["error_index"] != 0].copy()
    total_error_prob = error_classes["error_prob"].sum()
    error_classes["calibrated_prob"] = error_classes["error_prob"] / total_error_prob
    
    # Sort by calibrated probability (descending) and assign calibrated ranks
    error_classes = error_classes.sort_values("calibrated_prob", ascending=False).reset_index(drop=True)
    error_classes["calibrated_rank"] = range(len(error_classes))
    
    # Compute cumulative probability
    error_classes["cumulative_prob"] = error_classes["calibrated_prob"].cumsum()
    
    # Filter eligible classes: cumulative_prob <= coverage_fraction
    eligible_mask = error_classes["cumulative_prob"] <= coverage_fraction
    # Always include at least the most probable error class
    if not eligible_mask.any():
        eligible_mask.iloc[0] = True
    eligible_classes = error_classes[eligible_mask]
    
    num_eligible = len(eligible_classes)
    num_to_sample = min(n_classes - 1, num_eligible)  # -1 because initial best is no-error
    
    # Special case: deterministic (no sampling needed)
    if num_to_sample >= num_eligible:
        # Include all eligible error classes + no-error
        all_llrs = np.concatenate([[initial_best_llr], eligible_classes["fixed_llr"].values])
        if len(all_llrs) >= 2:
            sorted_llrs = np.sort(all_llrs)
            gap = sorted_llrs[1] - sorted_llrs[0]
        else:
            gap = 0.0
        return np.full(n_reps, gap)
    
    # Monte Carlo simulation (random sampling needed)
    gap_proxies = np.zeros(n_reps)
    
    for rep in range(n_reps):
        # Sample indices from eligible error classes
        sampled_indices = rng.choice(num_eligible, size=num_to_sample, replace=False)
        sampled_llrs = eligible_classes.iloc[sampled_indices]["fixed_llr"].values
        # Combine with initial best (no-error class)
        all_llrs = np.concatenate([[initial_best_llr], sampled_llrs])
        
        # Compute gap proxy
        sorted_llrs = np.sort(all_llrs)
        gap_proxies[rep] = sorted_llrs[1] - sorted_llrs[0]
    
    return gap_proxies


def compute_true_gap(shot_data: pd.DataFrame) -> float:
    """
    Compute the true logical gap from exhaustive exploration data.
    
    Parameters
    ----------
    shot_data : pd.DataFrame
        Data for a single shot with column: fixed_llr
    
    Returns
    -------
    true_gap : float
        True logical gap (second-best LLR - best LLR)
    """
    sorted_llrs = shot_data["fixed_llr"].sort_values().values
    if len(sorted_llrs) >= 2:
        return sorted_llrs[1] - sorted_llrs[0]
    return 0.0


print("Simulation functions defined.")
print("  - Initial best class: no-error (error_index=0)")
print("  - Sampling pool: calibrated distribution (error classes only)")
print("  - Special case: deterministic when N_classes covers all eligible classes")

Simulation functions defined.
  - Initial best class: no-error (error_index=0)
  - Sampling pool: calibrated distribution (error classes only)
  - Special case: deterministic when N_classes covers all eligible classes


In [35]:
# ==============================================================================
# Run Monte Carlo Grid Simulation (with stop-and-restart support)
# ==============================================================================

from tqdm.auto import tqdm

# Parameter grid
N_CLASSES_VALUES = 2**np.arange(3, 13)
COVERAGE_FRACTION_VALUES = np.arange(0.1, 1.01, 0.1)
N_MC_REPS = 50

# Output file for raw MC samples (enables stop-and-restart)
RAW_MC_OUTPUT_PATH = RESULTS_PATH.parent / f"bb_n{N_QUBITS}_p{P}_random_proxy_raw.parquet"

# Random seed for reproducibility
rng = np.random.default_rng(42)

print("Monte Carlo Simulation Configuration:")
print(f"  N_classes values: {N_CLASSES_VALUES}")
print(f"  Coverage fractions: {len(COVERAGE_FRACTION_VALUES)} values from {COVERAGE_FRACTION_VALUES[0]:.1f} to {COVERAGE_FRACTION_VALUES[-1]:.1f}")
print(f"  MC repetitions per config: {N_MC_REPS}")
print(f"  Raw output file: {RAW_MC_OUTPUT_PATH.name}")
print()

# Check for existing data (stop-and-restart support)
all_raw_results = []
completed_shots = set()

if RAW_MC_OUTPUT_PATH.exists():
    existing_df = pd.read_parquet(RAW_MC_OUTPUT_PATH)
    all_raw_results = existing_df.to_dict("records")
    completed_shots = set(existing_df["shot_id"].unique())
    print(f"Resuming from existing file: {len(existing_df):,} records, {len(completed_shots)} shots completed")

# Determine which shots to process
shot_groups = [(sid, sdata) for sid, sdata in df_results.groupby("shot_id")]
remaining_shots = [(sid, sdata) for sid, sdata in shot_groups if sid not in completed_shots]

if not remaining_shots:
    print("All shots already processed. Loading existing data.")
    df_mc_raw = pd.DataFrame(all_raw_results)
else:
    print(f"Processing {len(remaining_shots)} remaining shots (out of {len(shot_groups)} total)...")
    
    total_iterations = len(remaining_shots) * len(COVERAGE_FRACTION_VALUES) * len(N_CLASSES_VALUES)
    
    with tqdm(total=total_iterations, desc="Simulating") as pbar:
        for shot_id, shot_data in remaining_shots:
            # Compute true gap for this shot
            true_gap = compute_true_gap(shot_data)
            
            shot_records = []
            for coverage_frac in COVERAGE_FRACTION_VALUES:
                for n_classes in N_CLASSES_VALUES:
                    # Simulate random gap proxy
                    gap_proxies = simulate_random_gap_proxy(
                        shot_data, n_classes, coverage_frac, N_MC_REPS, rng
                    )
                    
                    # Store each MC sample as a separate row
                    for mc_rep, gap_proxy in enumerate(gap_proxies):
                        shot_records.append({
                            "shot_id": shot_id,
                            "n_classes": int(n_classes),
                            "coverage_frac": float(coverage_frac),
                            "mc_rep": mc_rep,
                            "true_gap": float(true_gap),
                            "gap_proxy": float(gap_proxy),
                        })
                    pbar.update(1)
            
            # Add to results and save incrementally
            all_raw_results.extend(shot_records)
            pd.DataFrame(all_raw_results).to_parquet(RAW_MC_OUTPUT_PATH, index=False)
    
    df_mc_raw = pd.DataFrame(all_raw_results)
    print(f"\nSimulation complete. Raw data saved to: {RAW_MC_OUTPUT_PATH.name}")

print(f"\nRaw data: {len(df_mc_raw):,} records ({df_mc_raw['shot_id'].nunique()} shots)")
print(f"\nPreview:")
df_mc_raw.head(10)

Monte Carlo Simulation Configuration:
  N_classes values: [   8   16   32   64  128  256  512 1024 2048 4096]
  Coverage fractions: 10 values from 0.1 to 1.0
  MC repetitions per config: 50
  Raw output file: bb_n144_p0.003_random_proxy_raw.parquet

Processing 563 remaining shots (out of 563 total)...


Simulating:   0%|          | 0/56300 [00:00<?, ?it/s]


Simulation complete. Raw data saved to: bb_n144_p0.003_random_proxy_raw.parquet

Raw data: 2,815,000 records (563 shots)

Preview:


,shot_id,n_classes,coverage_frac,mc_rep,true_gap,gap_proxy
0,0,8,0.100000,0,18.510165,65.725986
1,0,8,0.100000,1,18.510165,28.212727
2,0,8,0.100000,2,18.510165,65.725986
3,0,8,0.100000,3,18.510165,65.725986
4,0,8,0.100000,4,18.510165,82.571773
5,0,8,0.100000,5,18.510165,24.624800
6,0,8,0.100000,6,18.510165,28.212727
7,0,8,0.100000,7,18.510165,24.624800
8,0,8,0.100000,8,18.510165,55.798335
9,0,8,0.100000,9,18.510165,28.212727


In [54]:
# ==============================================================================
# Scatter Plot: Gap Proxy vs True Gap for Specific Configuration(s)
# ==============================================================================

# Configuration to visualize (can be single value or list)
PLOT_N_CLASSES = [16, 128]
PLOT_COVERAGE_FRAC = [0.3, 1.0]

# Convert to lists if single values
n_classes_list = [PLOT_N_CLASSES] if not isinstance(PLOT_N_CLASSES, list) else PLOT_N_CLASSES
coverage_list = [PLOT_COVERAGE_FRAC] if not isinstance(PLOT_COVERAGE_FRAC, list) else PLOT_COVERAGE_FRAC

n_rows = len(coverage_list)
n_cols = len(n_classes_list)

# Create subplot titles
subplot_titles = [
    f"N={nc}, cov={cov}" 
    for cov in coverage_list 
    for nc in n_classes_list
]

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
)

# Determine global axis range for consistency
max_val = max(df_mc_raw["true_gap"].max(), df_mc_raw["gap_proxy"].max()) * 1.05

for row_idx, coverage_frac in enumerate(coverage_list):
    for col_idx, n_classes in enumerate(n_classes_list):
        # Filter data for this configuration
        mask = (df_mc_raw["n_classes"] == n_classes) & (
            np.isclose(df_mc_raw["coverage_frac"], coverage_frac, atol=0.01)
        )
        subset = df_mc_raw[mask]
        
        # Subsample if too many points
        if len(subset) > 5000:
            subset = subset.sample(n=5000, random_state=42)
        
        # Compute correlation
        corr, _ = stats.spearmanr(subset["true_gap"], subset["gap_proxy"])
        
        # Scatter plot
        fig.add_trace(
            go.Scatter(
                x=subset["true_gap"],
                y=subset["gap_proxy"],
                mode="markers",
                marker=dict(size=4, color="steelblue", opacity=0.4),
                showlegend=False,
                hovertemplate="Shot %{customdata}<br>True: %{x:.1f}<br>Proxy: %{y:.1f}<extra></extra>",
                customdata=subset["shot_id"],
            ),
            row=row_idx + 1,
            col=col_idx + 1,
        )
        
        # Diagonal line (perfect prediction)
        fig.add_trace(
            go.Scatter(
                x=[0, max_val],
                y=[0, max_val],
                mode="lines",
                line=dict(color="red", dash="dash", width=1),
                showlegend=False,
            ),
            row=row_idx + 1,
            col=col_idx + 1,
        )
        
        # Add correlation annotation
        fig.add_annotation(
            text=f"ρ={corr:.2f}",
            xref=f"x{row_idx * n_cols + col_idx + 1}" if row_idx * n_cols + col_idx > 0 else "x",
            yref=f"y{row_idx * n_cols + col_idx + 1}" if row_idx * n_cols + col_idx > 0 else "y",
            x=max_val * 0.95,
            y=max_val * 0.05,
            showarrow=False,
            font=dict(size=10),
            bgcolor="white",
        )
        
        # Set axis ranges
        fig.update_xaxes(range=[0, max_val], row=row_idx + 1, col=col_idx + 1)
        fig.update_yaxes(range=[0, max_val], row=row_idx + 1, col=col_idx + 1)

# Add axis labels only on edges
for col_idx in range(n_cols):
    fig.update_xaxes(title_text="True Gap", row=n_rows, col=col_idx + 1)
for row_idx in range(n_rows):
    fig.update_yaxes(title_text="Gap Proxy", row=row_idx + 1, col=1)

fig.update_layout(
    title="Gap Proxy vs True Gap (Each Point = One MC Sample)",
    height=350 * n_rows,
    width=350 * n_cols,
    showlegend=False,
)

fig.show()

In [48]:
# ==============================================================================
# Compute Aggregated Statistics from Raw MC Data
# ==============================================================================

# Compute derived metrics for each sample
df_mc_raw["gap_ratio"] = df_mc_raw["gap_proxy"] / df_mc_raw["true_gap"]
df_mc_raw["abs_error"] = np.abs(df_mc_raw["gap_proxy"] - df_mc_raw["true_gap"])

# Aggregate per (shot_id, n_classes, coverage_frac)
df_mc_results = df_mc_raw.groupby(["shot_id", "n_classes", "coverage_frac"]).agg(
    true_gap=("true_gap", "first"),
    gap_proxy_mean=("gap_proxy", "mean"),
    gap_proxy_std=("gap_proxy", "std"),
    gap_proxy_min=("gap_proxy", "min"),
    gap_proxy_max=("gap_proxy", "max"),
    gap_ratio_mean=("gap_ratio", "mean"),
    gap_ratio_std=("gap_ratio", "std"),
    gap_ratio_min=("gap_ratio", "min"),
    gap_ratio_max=("gap_ratio", "max"),
    abs_error_mean=("abs_error", "mean"),
    abs_error_std=("abs_error", "std"),
    abs_error_min=("abs_error", "min"),
    abs_error_max=("abs_error", "max"),
).reset_index()

# Aggregate across shots for visualization
df_aggregated = df_mc_results.groupby(["n_classes", "coverage_frac"]).agg({
    "gap_ratio_mean": "mean",
    "gap_ratio_std": "mean",  # Average of per-shot stds
    "abs_error_mean": "mean",
    "abs_error_std": "mean",  # Average of per-shot stds
}).reset_index()

print("Aggregated Statistics:")
print(f"  Per-config results: {len(df_mc_results):,} rows")
print(f"  Cross-shot aggregated: {len(df_aggregated):,} rows")
print(f"\ndf_mc_results preview:")
df_mc_results.head(10)

Aggregated Statistics:
  Per-config results: 56,300 rows
  Cross-shot aggregated: 100 rows

df_mc_results preview:


,shot_id,n_classes,coverage_frac,true_gap,gap_proxy_mean,gap_proxy_std,gap_proxy_min,gap_proxy_max,gap_ratio_mean,gap_ratio_std,gap_ratio_min,gap_ratio_max,abs_error_mean,abs_error_std,abs_error_min,abs_error_max
0,0,8,0.100000,18.510165,51.161286,29.219450,24.624800,118.304015,2.763956,1.578562,1.330339,6.391300,32.651122,29.219450,6.114636,99.793851
1,0,8,0.200000,18.510165,62.477526,26.895358,24.624800,98.431126,3.375309,1.453005,1.330339,5.317680,43.967361,26.895358,6.114636,79.920962
2,0,8,0.300000,18.510165,76.972959,30.886790,24.624800,145.586710,4.158416,1.668639,1.330339,7.865230,58.462794,30.886790,6.114636,127.076545
3,0,8,0.400000,18.510165,75.469925,28.815236,24.624800,122.455852,4.077215,1.556725,1.330339,6.615600,56.959760,28.815236,6.114636,103.945688
4,0,8,0.500000,18.510165,79.534758,29.560765,38.111464,158.970855,4.296815,1.597002,2.058948,8.588300,61.024593,29.560765,19.601299,140.460690
5,0,8,0.600000,18.510165,80.239569,28.528443,28.212727,136.943416,4.334892,1.541231,1.524175,7.398282,61.729405,28.528443,9.702562,118.433251
6,0,8,0.700000,18.510165,84.560656,26.708478,28.212727,143.831883,4.568336,1.442909,1.524175,7.770427,66.050491,26.708478,9.702562,125.321719
7,0,8,0.800000,18.510165,81.148938,29.663896,24.624800,137.767365,4.384020,1.602573,1.330339,7.442795,62.638773,29.663896,6.114636,119.257200
8,0,8,0.900000,18.510165,94.901064,22.939991,50.892833,146.147138,5.126970,1.239319,2.749453,7.895507,76.390899,22.939991,32.382669,127.636973
9,0,8,1.000000,18.510165,95.247916,26.093544,33.719916,133.950261,5.145709,1.409687,1.821697,7.236579,76.737752,26.093544,15.209751,115.440096


In [49]:
# ==============================================================================
# Scale Dependency Analysis: Gap Ratio vs Absolute Error
# ==============================================================================
# 
# A good metric should be scale-independent, meaning its value shouldn't 
# systematically depend on the magnitude of the true gap. This allows fair
# comparison across shots with different true gap sizes.
#
# We analyze:
# - Correlation between true_gap and gap_ratio_mean across shots
# - Correlation between true_gap and abs_error_mean across shots
#
# If abs_error shows strong positive correlation with true_gap but gap_ratio
# doesn't, then gap_ratio is the preferred scale-invariant metric.

from plotly.subplots import make_subplots

# Aggregate per-shot statistics (average across all configurations for each shot)
df_per_shot = df_mc_results.groupby("shot_id").agg({
    "true_gap": "first",  # Same for all configs within a shot
    "gap_ratio_mean": "mean",
    "abs_error_mean": "mean",
}).reset_index()

# Compute correlations
corr_ratio_spearman, p_ratio_spearman = stats.spearmanr(
    df_per_shot["true_gap"], df_per_shot["gap_ratio_mean"]
)
corr_ratio_pearson, p_ratio_pearson = stats.pearsonr(
    df_per_shot["true_gap"], df_per_shot["gap_ratio_mean"]
)

corr_abs_spearman, p_abs_spearman = stats.spearmanr(
    df_per_shot["true_gap"], df_per_shot["abs_error_mean"]
)
corr_abs_pearson, p_abs_pearson = stats.pearsonr(
    df_per_shot["true_gap"], df_per_shot["abs_error_mean"]
)

print("=" * 70)
print("SCALE DEPENDENCY ANALYSIS")
print("=" * 70)
print("\nCorrelation with True Gap (across shots, averaged over configurations):")
print("-" * 70)
print(f"  Gap Ratio:      Spearman ρ = {corr_ratio_spearman:+.4f} (p = {p_ratio_spearman:.2e})")
print(f"                  Pearson r  = {corr_ratio_pearson:+.4f} (p = {p_ratio_pearson:.2e})")
print(f"  Absolute Error: Spearman ρ = {corr_abs_spearman:+.4f} (p = {p_abs_spearman:.2e})")
print(f"                  Pearson r  = {corr_abs_pearson:+.4f} (p = {p_abs_pearson:.2e})")

# Interpretation
print("\n" + "-" * 70)
print("INTERPRETATION:")
print("-" * 70)
if abs(corr_abs_spearman) > abs(corr_ratio_spearman) + 0.1:
    print("  Absolute error shows STRONGER correlation with true gap than gap ratio.")
    print("  → Gap ratio is more scale-invariant and preferred for cross-shot comparison.")
    preferred_metric = "gap_ratio"
elif abs(corr_ratio_spearman) > abs(corr_abs_spearman) + 0.1:
    print("  Gap ratio shows STRONGER correlation with true gap than absolute error.")
    print("  → Absolute error is more scale-invariant (unexpected).")
    preferred_metric = "abs_error"
else:
    print("  Both metrics show similar correlation strength with true gap.")
    print("  → Either metric can be used; gap ratio is typically preferred for interpretability.")
    preferred_metric = "gap_ratio"

# Create side-by-side scatter plots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"Gap Ratio vs True Gap (ρ = {corr_ratio_spearman:.3f})",
        f"Absolute Error vs True Gap (ρ = {corr_abs_spearman:.3f})"
    ],
    horizontal_spacing=0.12,
)

# Gap Ratio scatter
fig.add_trace(
    go.Scatter(
        x=df_per_shot["true_gap"],
        y=df_per_shot["gap_ratio_mean"],
        mode="markers",
        marker=dict(color="steelblue", size=12, line=dict(width=1, color="black")),
        text=[f"Shot {int(s)}" for s in df_per_shot["shot_id"]],
        hovertemplate="%{text}<br>True Gap: %{x:.2f}<br>Gap Ratio: %{y:.3f}<extra></extra>",
        showlegend=False,
    ),
    row=1, col=1
)

# Add trend line for gap ratio
z_ratio = np.polyfit(df_per_shot["true_gap"], df_per_shot["gap_ratio_mean"], 1)
p_ratio = np.poly1d(z_ratio)
x_range = np.linspace(df_per_shot["true_gap"].min(), df_per_shot["true_gap"].max(), 100)
fig.add_trace(
    go.Scatter(
        x=x_range,
        y=p_ratio(x_range),
        mode="lines",
        line=dict(color="red", dash="dash", width=2),
        showlegend=False,
    ),
    row=1, col=1
)

# Absolute Error scatter
fig.add_trace(
    go.Scatter(
        x=df_per_shot["true_gap"],
        y=df_per_shot["abs_error_mean"],
        mode="markers",
        marker=dict(color="darkorange", size=12, line=dict(width=1, color="black")),
        text=[f"Shot {int(s)}" for s in df_per_shot["shot_id"]],
        hovertemplate="%{text}<br>True Gap: %{x:.2f}<br>Abs Error: %{y:.2f}<extra></extra>",
        showlegend=False,
    ),
    row=1, col=2
)

# Add trend line for absolute error
z_abs = np.polyfit(df_per_shot["true_gap"], df_per_shot["abs_error_mean"], 1)
p_abs = np.poly1d(z_abs)
fig.add_trace(
    go.Scatter(
        x=x_range,
        y=p_abs(x_range),
        mode="lines",
        line=dict(color="red", dash="dash", width=2),
        showlegend=False,
    ),
    row=1, col=2
)

fig.update_xaxes(title_text="True Gap", row=1, col=1)
fig.update_xaxes(title_text="True Gap", row=1, col=2)
fig.update_yaxes(title_text="Mean Gap Ratio", row=1, col=1)
fig.update_yaxes(title_text="Mean Absolute Error", row=1, col=2)

fig.update_layout(
    title="Scale Dependency: Metric vs True Gap (Each Point = One Shot)",
    height=400,
    showlegend=False,
)

fig.show()

print(f"\n→ Preferred metric for visualization: {preferred_metric.upper().replace('_', ' ')}")

SCALE DEPENDENCY ANALYSIS

Correlation with True Gap (across shots, averaged over configurations):
----------------------------------------------------------------------
  Gap Ratio:      Spearman ρ = -0.7506 (p = 4.75e-103)
                  Pearson r  = -0.4524 (p = 9.47e-30)
  Absolute Error: Spearman ρ = -0.1832 (p = 1.22e-05)
                  Pearson r  = -0.1977 (p = 2.28e-06)

----------------------------------------------------------------------
INTERPRETATION:
----------------------------------------------------------------------
  Gap ratio shows STRONGER correlation with true gap than absolute error.
  → Absolute error is more scale-invariant (unexpected).



→ Preferred metric for visualization: ABS ERROR


In [52]:
# ==============================================================================
# Heatmap: Mean Absolute Error vs (N_classes, coverage_fraction)
# ==============================================================================

# Pivot for heatmap
pivot_abs_error = df_aggregated.pivot(
    index="coverage_frac", columns="n_classes", values="abs_error_mean"
)

# Create heatmap
fig = go.Figure(data=go.Heatmap(
    z=pivot_abs_error.values,
    x=[str(x) for x in pivot_abs_error.columns],
    y=[f"{y:.1f}" for y in pivot_abs_error.index],
    colorscale="YlOrRd",
    reversescale=True,  # Lower error = greener
    zmin=0,
    text=np.round(pivot_abs_error.values, 1),
    texttemplate="%{text}",
    textfont={"size": 10},
    hovertemplate="N_classes: %{x}<br>Coverage: %{y}<br>Abs Error: %{z:.2f}<extra></extra>",
))

fig.update_layout(
    title="Mean Absolute Error |Gap Proxy - True Gap| by Parameters",
    xaxis_title="N_classes",
    yaxis_title="Coverage Fraction",
    height=450,
)

fig.show()

# Print summary
print("\nMean Absolute Error by (N_classes, coverage_fraction):")
print(pivot_abs_error.round(2).to_string())


Mean Absolute Error by (N_classes, coverage_fraction):
n_classes           8          16        32        64        128       256       512       1024      2048      4096
coverage_frac                                                                                                      
0.100000       70.790000  45.550000 35.600000 35.600000 35.600000 35.600000 35.600000 35.600000 35.600000 35.600000
0.200000       71.090000  46.450000 29.990000 20.560000 20.560000 20.560000 20.560000 20.560000 20.560000 20.560000
0.300000       71.190000  46.040000 28.880000 16.650000 10.630000 10.630000 10.630000 10.630000 10.630000 10.630000
0.400000       73.860000  47.790000 30.260000 17.280000  7.590000  5.550000  5.550000  5.550000  5.550000  5.550000
0.500000       86.090000  57.900000 38.170000 23.720000 12.570000  3.630000  3.230000  3.230000  3.230000  3.230000
0.600000       98.840000  69.580000 48.190000 32.130000 19.570000  9.500000  1.820000  1.820000  1.820000  1.820000
0.700000      10

In [53]:
# ==============================================================================
# Convergence Curves: Absolute Error vs N_classes for each coverage_fraction
# ==============================================================================

fig = go.Figure()

# Color palette for different coverage fractions
colors = px.colors.qualitative.Plotly

for i, coverage_frac in enumerate(sorted(COVERAGE_FRACTION_VALUES)):
    subset = df_aggregated[df_aggregated["coverage_frac"] == coverage_frac]
    subset = subset.sort_values("n_classes")
    
    color = colors[i % len(colors)]
    
    # Error band (mean ± std)
    x_vals = subset["n_classes"].values
    y_mean = subset["abs_error_mean"].values
    y_std = subset["abs_error_std"].values
    y_upper = y_mean + y_std
    y_lower = np.maximum(y_mean - y_std, 0)  # Clamp to 0 (error can't be negative)
    
    # Create filled area for error band
    fig.add_trace(go.Scatter(
        x=np.concatenate([x_vals, x_vals[::-1]]),
        y=np.concatenate([y_upper, y_lower[::-1]]),
        fill="toself",
        fillcolor=color,
        opacity=0.2,
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip",
        name=f"coverage={coverage_frac:.1f} (band)",
    ))
    
    # Mean line with markers
    fig.add_trace(go.Scatter(
        x=x_vals,
        y=y_mean,
        mode="lines+markers",
        name=f"coverage={coverage_frac:.1f}",
        line=dict(color=color, width=2),
        marker=dict(size=8),
    ))

# Add horizontal line at y=0 (perfect approximation)
fig.add_hline(
    y=0.0,
    line_dash="dash",
    line_color="black",
    annotation_text="Perfect (error=0)",
    annotation_position="top right",
)

fig.update_layout(
    title="Convergence of Absolute Error to Zero (Mean ± Std)",
    xaxis_title="N_classes",
    yaxis_title="Absolute Error |Gap Proxy - True Gap|",
    xaxis_type="log",
    yaxis=dict(rangemode="tozero"),
    legend_title="Coverage Fraction",
    height=500,
)

fig.show()